![Practicum AI Logo image](https://github.com/PracticumAI/practicumai.github.io/blob/main/images/logo/PracticumAI_logo_250x50.png?raw=true) <img src="https://github.com/PracticumAI/practicumai.github.io/blob/84b04be083ca02e5c7e92850f9afd391fc48ae2a/images/icons/practicumai_computer_vision.png?raw=true" alt="Practicum AI: Computer Vision icon" align="right" width=50>
***

# Transfer Learning Concepts

You may recall *Practicum AI*"s heroine Amelia, the AI-savvy nutritionist. At the end of our *[Deep Learning Foundations course](https://practicumai.org/courses/deep_learning/)*, Amelia was helping with a computer vision project. If only she had known about transfer learning, it could have saved her a lot of time! In this notebook, we will get some hands-on experience with transfer learning and show you how to use it to improve your workflows.

![Figure 2 of the AgriNet paper used as the cover image for this notebook. Figure 2 depicts using transfer learning to make a computer vision model more efficient](images/agrinet_figure-cover.jpg)


## AI Pathway review for Transfer Learning & AgriNet 

If you have taken our [Getting Started with AI course](https://practicumai.org/courses/getting_started/), you may remember this figure of the AI Application Development Pathway. Let's take a quick review of how we will apply this to our case study of AgriNet and it's use of transfer learning.

![AI Application Development Pathway image showing the 7 steps in developing an AI application](https://practicumai.org/getting_started/images/application_dev_pathway.png)

1. **Choose a problem to solve:** In this example, we will be trying to make a computer vision model that can recognize images of plants. 
2. **Gather data:** The data for the example comes from [HuggingFace](https://www.huggingface.co//), a great repository of datasets, code, and models.
3. **Clean and prepare the data:** In the *Deep Learning Foundations* course, we assumed that this was done for us. One issue that we ran into was that of class imbalance. Here, the (probably very tired) researchers that created the AgriNet dataset have already balanced the classes for us!
4. **Choose a model:** In the *Deep Learning Foundations* course, we presented the model with little detail. Here, we will be using a pre-trained model, ResNet50, and applying transfer learning to it.
   * In the step where you'd choose a model, one can approach this in two ways:
      * **Train from scratch:** This is where you start with a randomly initialized model and train it on your data. This can be computationally expensive and time-consuming.
      * **Transfer learning:** This is where you start with a pre-trained model and fine-tune it on your data. This is often faster and requires less data.
5. **Train the model:** We'll be comparing training a model from scratch to transfering to a pre-trained model. With so many hyperparameters to tune and compare, it's easy to lose track of what combinations have been tried and how changes impacted model performance. 
   * In this notebook, we introduce you to [TensorBoard](https://www.tensorflow.org/tensorboard), one popular tool in a class of tools known as **experiment tracking** or **MLOps (Machine learning operations) tools**. These tools help track changes to hyperparameters, the training process, and the data. They allow comparison among runs and can even automate multiple runs for you. Learning to use MLOps tools will help you as you continue to learn more about AI workflows.
   * We'll demonstrate three approaches in this notebook:
      - Training a baseline model from scratch.
      - Fine-tuning a model pre-trained on ImageNet.
      - Fine-tuning a model pre-trained on AgriNet, a domain-specific dataset.
6. **Evaluate the model:** We will use the metrics we gather to make decisions about the model. 
7. **Deploy the model:** We won't get to this stage in this exercise, but hopefully, we will end up with a model that could be deployed and achieve relatively good accuracy at solving the problem.


### A Refresher

If you need a refresher, or havent taken the *Deep Learning Foundations* course, the final notebook is part of this repository: [DLF_03_bees_vs_wasps.ipynb](DLF_03_bees_vs_wasps.ipynb).

### A Quick Primer on the Baseline Model
We'll train a simple convolutional neural network (CNN) from scratch as a baseline for comparison.

We'll define a CNN with basic layers, such as convolutional, pooling, and fully connected layers. The model will be compiled with the Adam optimizer and categorical cross-entropy loss, then trained on the dataset. Strictly speaking, a thorough knowledge of CNNs is not required for this notebook, but if you're interested in learning more, we recommend the our [PracticumAI: Computer Vision](https://github.com/PracticumAI/computer_vision) Intermediate course.

That said, with *any* machine learning work, the better you understand the model, the better you can tune it to your needs.


### Transfer Learning with ImageNet

We'll use the VGG19 model pre-trained on ImageNet and fine-tune it for plant disease detection.

ImageNet pre-trained models have learned general features (e.g., edges, textures) that can be adapted to our specific task. This significantly reduces the training time and data requirements.

The base layers of VGG19 will be frozen to retain their pre-trained features. We'll add custom layers for classification and fine-tune the model on our dataset.

## 1. Import the libraries we will use

In [2]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from PIL import ImageFile
import requests
import zipfile
import matplotlib.pyplot as plt

## 2. Jupyter magic commands

In Jupyter, the `%` is used as a "magic" command. These extend Python in various ways. In this case, Tensorboad functionality is added using `%load_ext tensorboard`

Again, [Tensorboard](https://www.tensorflow.org/tensorboard) is the tool we"ll use for experiment tracking in many of these notebooks. 

In [3]:
# Load the TensorBoard notebook extension
%load_ext tensorboard

## 3. Getting the data

Gotta have data to train a model! The code below downloads the curated version of the dataset and unzips it.

In [4]:
# Download the dataset, extract it to the data folder and remove the zip file
download_path = "https://data.rc.ufl.edu/pub/practicum-ai/Transfer_Learning_Intermediate/agrinet_curated.zip"
zip_path = "data/agrinet_curated.zip"
data_path = "data"

# Create the data directory if it does not exist
if not os.path.exists(data_path):
    os.makedirs(data_path)

# Download the zip file
r = requests.get(download_path)
with open(zip_path, "wb") as f:
    f.write(r.content)

# Extract the zip file
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(data_path)

# Remove the zip file
os.remove(zip_path)

# Paths to dataset
train_dir = "data/agri_net_train"
val_dir = "data/agri_net_val"
test_dir = "data/agri_net_test"

## 4. Preparing the generators

Data generators are used to load data in batches and preprocess it. We'll use the `ImageDataGenerator` class from Keras to load and preprocess the data.

In [5]:
# Define data generators
datagen = ImageDataGenerator(rescale=1.0 / 255, validation_split=0.2)

train_gen = datagen.flow_from_directory(
    train_dir, target_size=(224, 224), batch_size=32, subset="training"
)

val_gen = datagen.flow_from_directory(
    val_dir, target_size=(224, 224), batch_size=32, subset="validation"
)

Found 50502 images belonging to 86 classes.
Found 1768 images belonging to 86 classes.


### What, Why, and How: Transfer Learning with AgriNet

**What:**
We'll use the VGG19 model pre-trained on the AgriNet dataset, which is domain-specific to agriculture.

**Why:**
Domain-specific pre-training captures features relevant to agricultural tasks, such as plant patterns and disease characteristics, which can further improve model performance compared to generic pre-trained models.

**How:**
Similar to the ImageNet approach, we'll freeze the base layers of the AgriNet model, add custom classification layers, and fine-tune the model on our dataset.

## Baseline Model

We'll train a simple convolutional neural network from scratch and use it as our baseline for performance comparison.

### What, Why, and How: Performance Comparison

**What:**
We'll compare the performance of the three models (baseline, ImageNet pre-trained, and AgriNet pre-trained) using metrics like accuracy and F1-score.

**Why:**
This step helps quantify the benefits of transfer learning and highlights the impact of using domain-specific pre-trained models.

**How:**
We'll evaluate each model on the test set and visualize the results using performance metrics and charts.

In [6]:
# Handle truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Define baseline model
baseline_model = Sequential(
    [
        Conv2D(32, (3, 3), activation="relu", input_shape=(224, 224, 3)),
        MaxPooling2D(2, 2),
        Flatten(),
        Dense(128, activation="relu"),
        Dropout(0.5),
        Dense(len(train_gen.class_indices), activation="softmax"),
    ]
)

baseline_model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

# Train the baseline model
history_baseline = baseline_model.fit(train_gen, validation_data=val_gen, epochs=10)

C:\Users\sushi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10


C:\Users\sushi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1579/1579 ━━━━━━━━━━━━━━━━━━━━ 2127s 1s/step - accuracy: 0.1369 - loss: 4.4592 - val_accuracy: 0.2602 - val_loss: 2.9042
Epoch 2/10
1579/1579 ━━━━━━━━━━━━━━━━━━━━ 1004s 632ms/step - accuracy: 0.2270 - loss: 3.0906 - val_accuracy: 0.3416 - val_loss: 2.5120
Epoch 3/10
1579/1579 ━━━━━━━━━━━━━━━━━━━━ 873s 551ms/step - accuracy: 0.2632 - loss: 2.8591 - val_accuracy: 0.3688 - val_loss: 2.4328
Epoch 4/10
1579/1579 ━━━━━━━━━━━━━━━━━━━━ 831s 525ms/step - accuracy: 0.2907 - loss: 2.6928 - val_accuracy: 0.4129 - val_loss: 2.1760
Epoch 5/10
1579/1579 ━━━━━━━━━━━━━━━━━━━━ 858s 542ms/step - accuracy: 0.3196 - loss: 2.5566 - val_accuracy: 0.4395 - val_loss: 2.0135
Epoch 6/10
1579/1579 ━━━━━━━━━━━━━━━━━━━━ 825s 521ms/step - accuracy: 0.3633 - loss: 2.3616 - val_accuracy: 0.4615 - val_loss: 1.9593
Epoch 7/10
1579/1579 ━━━━━━━━━━━━━━━━━━━━ 798s 505ms/step - accuracy: 0.3973 - loss: 2.2294 - val_accuracy: 0.4796 - val_loss: 1.8365
Epoch 8/10
1579/1579 ━━━━━━━━━━━━━━━━━━━━ 791s 500ms/step - accuracy: 0.42

### Conclusion: Key Insights

- Transfer learning significantly improves performance compared to training from scratch, especially with limited data.
- Domain-specific pre-training (e.g., AgriNet) can further enhance accuracy and generalization for specialized tasks.
- These findings demonstrate the importance of transfer learning in tackling real-world challenges in agriculture.

## Transfer Learning with ImageNet

We'll use a pre-trained VGG19 model with ImageNet weights and fine-tune it on our dataset.

In [ ]:
from tensorflow.keras.applications import VGG19
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D

# Load pre-trained VGG19 model
imagenet_model = VGG19(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

# Freeze base layers
for layer in imagenet_model.layers:
    layer.trainable = False

# Add custom top layers
x = GlobalAveragePooling2D()(imagenet_model.output)
x = Dense(128, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(len(train_gen.class_indices), activation="softmax")(x)

imagenet_model = Model(inputs=imagenet_model.input, outputs=output)

imagenet_model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

# Train the model
history_imagenet = imagenet_model.fit(train_gen, validation_data=val_gen, epochs=10)

Epoch 1/10
 581/1579 ━━━━━━━━━━━━━━━━━━━━ 2:01:27 7s/step - accuracy: 0.2133 - loss: 3.5262

In [ ]:
from tensorflow.keras.applications import VGG19
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D

# Load pre-trained VGG19 model
imagenet_model = VGG19(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

# Freeze base layers
for layer in imagenet_model.layers:
    layer.trainable = False

# Add custom top layers
x = GlobalAveragePooling2D()(imagenet_model.output)
x = Dense(128, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(len(train_gen.class_indices), activation="softmax")(x)

imagenet_model = Model(inputs=imagenet_model.input, outputs=output)

imagenet_model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

# Train the model
history_imagenet = imagenet_model.fit(train_gen, validation_data=val_gen, epochs=10)

In [ ]:
# Assuming AgriNet weights are available locally
agri_weights_path = "/path/to/agri_vgg19_weights.h5"  # Replace with actual path

# Load the VGG19 model
agri_model = VGG19(weights=None, include_top=False, input_shape=(224, 224, 3))

# Load AgriNet weights
agri_model.load_weights(agri_weights_path)

# Freeze base layers
for layer in agri_model.layers:
    layer.trainable = False

# Add custom top layers
x = GlobalAveragePooling2D()(agri_model.output)
x = Dense(128, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(len(train_gen.class_indices), activation="softmax")(x)

agri_model = Model(inputs=agri_model.input, outputs=output)

agri_model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

# Train the model
history_agri = agri_model.fit(train_gen, validation_data=val_gen, epochs=10)

In [ ]:
# Assuming AgriNet weights are available locally
agri_weights_path = "/path/to/agri_vgg19_weights.h5"  # Replace with actual path

# Load the VGG19 model
agri_model = VGG19(weights=None, include_top=False, input_shape=(224, 224, 3))

# Load AgriNet weights
agri_model.load_weights(agri_weights_path)

# Freeze base layers
for layer in agri_model.layers:
    layer.trainable = False

# Add custom top layers
x = GlobalAveragePooling2D()(agri_model.output)
x = Dense(128, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(len(train_gen.class_indices), activation="softmax")(x)

agri_model = Model(inputs=agri_model.input, outputs=output)

agri_model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

# Train the model
history_agri = agri_model.fit(train_gen, validation_data=val_gen, epochs=10)

## Conclusion

In this notebook, we demonstrated the benefits of transfer learning in agricultural tasks. The AgriNet pre-trained model outperformed the ImageNet model and the baseline, showing the importance of domain-specific pre-training for specialized applications.